<a href="https://colab.research.google.com/github/aehaf988/Pro1/blob/main/project.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [24]:
#import and file paths
import math
import pandas as pd
import numpy as np
from scipy.special import gammaln
from scipy.optimize import minimize_scalar
import matplotlib.pyplot as plt

signature = '3' #large deletions with microhomology at breakpoints
#parameter: theta = SBS3 weight

In [25]:

RAW_URL = 'https://raw.githubusercontent.com/aehaf988/Pro1/main/Breast_genomes_mutational_catalog_96_subs_SBS3.txt'
catalog = pd.read_csv(RAW_URL, sep='\t', index_col=0, decimal=',')

print('Shape:', catalog.shape)
print('Dtypes:', catalog.dtypes.head())

# --- THE MISSING LINE ---
ref    = catalog['SBS3']
counts = catalog.drop(columns='SBS3')

print('Total mutations (samples only):', int(counts.values.sum()))
print('First mutation types:', list(catalog.index[:3]))
print('First samples:', list(counts.columns[:3]))     # note: counts, not catalog



Shape: (96, 120)
Dtypes: SBS3       object
PD3851a     int64
PD3890a     int64
PD3904a     int64
PD3905a     int64
dtype: object
Total mutations (samples only): 647692
First mutation types: ['A[C>A]A', 'A[C>A]C', 'A[C>A]G']
First samples: ['PD3851a', 'PD3890a', 'PD3904a']


In [26]:

def poisson_log_pmf(k, lam):
    return k * math.log(lam) - lam - math.lgamma(k + 1)

def NLL(theta, K_obs, N_total):
    lam = N_total * theta
    return -poisson_log_pmf(K_obs, lam)



K_obs   = 29
sample  = counts.columns[0]              # e.g. 'PD3851a'
k = counts[sample].values.astype(float)
N_total = 1000

result = minimize_scalar(NLL, bounds=(1e-9, 1.0),
                         method="bounded",
                         args=(K_obs, N_total))
print("theta =", result.x, "  -loglik =", result.fun)



def poisson_log_pmf(k, lam):
    """Vectorized log Poisson PMF. Works for scalar or array k, lam."""
    k   = np.asarray(k, dtype=float)
    lam = np.asarray(lam, dtype=float)
    log_lam = np.where(lam > 0, np.log(np.maximum(lam, 1e-300)), -np.inf)
    return k * log_lam - lam - gammaln(k + 1)

def NLL_signature(theta, k_obs, r, N_total):
    """Whole 96-channel fit against a single reference signature r."""
    k_obs = np.asarray(k_obs, dtype=float)
    r     = np.asarray(r,     dtype=float)
    lam   = N_total * theta * r
    return -poisson_log_pmf(k_obs, lam).sum()

r = (
    catalog['SBS3']
    .astype(str).str.replace(',', '.', regex=False)
    .astype(float).values
)
r = r / r.sum()                             # normalize
counts = catalog.drop(columns='SBS3').astype(int)

sample  = counts.columns[0]                 # e.g. 'PD3851a'
k       = counts[sample].values.astype(float)   # (96,)
N_total = int(k.sum())                      # per-sample total

print("ready:", sample, " N_total =", N_total,
      " k.shape =", k.shape, " r.shape =", r.shape, " r.sum() =", r.sum())

print("ready:", sample, "N_total =", N_total, "k.shape =", k.shape, "r.shape =", r.shape)





theta = 0.028999015183098803   -loglik = 2.6054599142824344
ready: PD3851a  N_total = 1561  k.shape = (96,)  r.shape = (96,)  r.sum() = 1.0
ready: PD3851a N_total = 1561 k.shape = (96,) r.shape = (96,)


In [31]:
# Split once, right after loading
ref    = catalog['SBS3'].astype(str).str.replace(',', '.').astype(float).values
counts = catalog.drop(columns='SBS3').astype(int)

# Now the total is over actual observations only
N_total = int(counts.values.sum())
print("N_total (observed mutations across all samples):", N_total)

N_per_sample = counts.sum(axis=0)      # Series: one total per column
sample  = counts.columns[0]
N_total = int(N_per_sample[sample])
print("N_total for", sample, "=", N_total)     # e.g. 1561

def simulate(theta, N_total, r, rng=None):
    if rng is None:
        rng = np.random.default_rng()
    r = np.asarray(r, float)
    r = r / r.sum()
    lam = N_total * theta * r          # (96,) expected counts
    return rng.poisson(lam)            # (96,) simulated counts

rng   = np.random.default_rng(0)
k_sim = simulate(theta=1.0, N_total=N_total, r=ref, rng=rng)
print("simulated counts:", k_sim[:5], "  sum:", k_sim.sum())


N_total (observed mutations across all samples): 647692
N_total for PD3851a = 1561
simulated counts: [35 13  7 25 35]   sum: 1598


In [28]:
#slope

def slope_at(score, theta, h):
  return (score(theta+h) -score(theta-h)) / (2*h)

k = counts[sample].values.astype(float)
N = int(k.sum())        # ← same as k.sum(), per-sample total

K = k.sum()
N = k.sum() if 'N' not in dir() else N   # only if you want Option A

for theta in [0.5, 1.0, 1.5]:
    num = slope_at(score, theta, h=1e-5)
    ana = N - K / theta
    print(f"theta={theta}  numeric={num:14.6f}  analytic={ana:14.6f}  diff={num-ana:.2e}")

def score(theta):
    return NLL_signature(theta, k, r, N)

for theta in [0.5, 1.0, 1.5]:
    num = slope_at(score, theta, h=1e-5)
    ana = N - K / theta
    print(f"theta={theta}  numeric={num:.6f}  analytic={ana:.6f}  "
          f"diff={num-ana:.2e}")

theta=0.5  numeric=  -1561.000000  analytic=  -1561.000000  diff=-4.13e-07
theta=1.0  numeric=     -0.000000  analytic=      0.000000  diff=-5.12e-08
theta=1.5  numeric=    520.333333  analytic=    520.333333  diff=-6.32e-09
theta=0.5  numeric=-1561.000000  analytic=-1561.000000  diff=-4.13e-07
theta=1.0  numeric=-0.000000  analytic=0.000000  diff=-5.12e-08
theta=1.5  numeric=520.333333  analytic=520.333333  diff=-6.32e-09


In [29]:
k = counts[sample].values.astype(float)
r = ref                                        # normalized
N = k.sum()

def deviance(k, r, N):
    exp = N * r
    mask = (k > 0) & (exp > 0)
    G2 = 2 * (k[mask] * np.log(k[mask] / exp[mask])).sum()
    df = mask.sum() - 1
    from scipy.stats import chi2
    return G2, df, 1 - chi2.cdf(G2, df)

G2, df, p = deviance(k, r, N)
print(f"Deviance G²={G2:.1f}  df={df}  p={p:.3g}")

Deviance G²=1430.0  df=95  p=0


In [23]:
# uncertainty: parametric bootstrap for one sample
K = k.sum()
theta_hat = 1.0
print("analytic SE:", theta_hat / np.sqrt(K))   # ≈ 0.025


A_LCG, C_LCG, M_LCG = 1664525, 1013904223, 2**32

def lcg(state, a, c, m):
    return (a * state + c) % m

def uniforms(n, seed):
    n = int(n)
    out = [0.0] * n
    state = int(seed) % M_LCG
    for i in range(n):
        state = lcg(state, A_LCG, C_LCG, M_LCG)
        out[i] = state / M_LCG
    return out

def choose(r, probs):
    running = 0.0
    for j in range(len(probs)):
        running += probs[j]
        if r < running:
            return j
    return len(probs) - 1

# ---------- inputs ----------
counts_vec = np.asarray(counts[counts.columns[0]], dtype=int)   # (96,) integer counts
N          = int(counts_vec.sum())                              # total mutations
events     = np.repeat(np.arange(96), counts_vec)               # (N,) values in [0..95]
B          = 1000
SEED       = 8

print("counts_vec length:", len(counts_vec))   # must be 96
print("N               :", N)                  # ~374–1500
print("events length   :", len(events))        # must equal N

# ---------- bootstrap ----------
rs    = uniforms(B * N, SEED)
equal = [1.0 / N] * N

boot_means = np.empty(B)
for b in range(B):
    resample = np.empty(N, dtype=int)
    for i in range(N):
        j = choose(rs[b * N + i], equal)
        resample[i] = events[j]               # <-- events, not values
    boot_means[b] = counts_vec[resample].mean()

sigma_boot = boot_means.std(ddof=1)
print("mean count per channel:", counts_vec.mean())
print("bootstrap SE         :", sigma_boot)

analytic SE: 0.025310373177627687


KeyboardInterrupt: 

In [30]:
#plots
#a. distribution of a signature across samples
#b. observed vs fitted spectrum for one sample
#c. per-signature weight across sample with CIs